In [ ]:
!pip install pm4py
import pandas as pd
import os, re, glob
from pm4py.objects.log.util import dataframe_utils
from pm4py.algo.discovery.dfg import algorithm as dfg_algorithm
from pm4py.objects.conversion.log import converter as log_converter
from collections import Counter, defaultdict
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.petri_net.obj import PetriNet, Marking   

In [ ]:
DATA_DIR = """/path/to/data/directory"""  # Update this path accordingly
OUTPUT_DIR = """/path/to/output/directory"""  # Update this path accordingly

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# DFG Abstraction

def pick(colnames, *aliases):
    low = {c.lower(): c for c in colnames}
    for a in aliases:
        if a.lower() in low:
            return low[a.lower()]
    return None

def compute_dfg_and_write(csv_path, out_dir):
    base = os.path.basename(csv_path)
    # Extract metadata from filename
    m = re.match(r"^([A-Za-z]+)-([A-Za-z]+)-([0-9.]+)-\d+\.csv$", base)
    if m:
        dat, typ, ratio = m.group(1), m.group(2), m.group(3)
        out_name = f"DFG_{dat}_{typ}_{ratio}.txt"
    else:
        out_name = f"DFG_{os.path.splitext(base)[0]}.txt"
    out_path = os.path.join(out_dir, out_name)

    # ---- 1) Load & map columns ----
    df = pd.read_csv(csv_path)
    case_col = pick(df.columns, "Case", "Case ID", "case", "caseid", "case:concept:name")
    act_col  = pick(df.columns, "Activity", "concept:name", "Task", "activity")
    ts_col   = pick(df.columns, "Timestamp", "time:timestamp", "EventTime", "time")
    assert case_col and act_col and ts_col, f"[{base}] Missing required columns (case/activity/timestamp)"

    df_pm4py = df.rename(columns={
        case_col: "case:concept:name",
        act_col:  "concept:name",
        ts_col:   "time:timestamp"
    })
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    df_pm4py = df_pm4py.dropna(subset=["time:timestamp"])
    df_pm4py = dataframe_utils.convert_timestamp_columns_in_df(df_pm4py)

    # ---- 2) Convert to EventLog ----
    event_log = log_converter.apply(df_pm4py)

    # ---- 3) Compute DFG ----
    dfg_freq = dfg_algorithm.apply(event_log, variant=dfg_algorithm.Variants.FREQUENCY)
    dfg_perf = dfg_algorithm.apply(event_log, variant=dfg_algorithm.Variants.PERFORMANCE)

    # ---- 4) Combine & save ----
    rows = []
    for (a, b), freq in dfg_freq.items():
        perf = dfg_perf.get((a, b), None)
        perf_str = f"{perf:.2f}" if isinstance(perf, (int, float)) and perf is not None else "NA"
        rows.append((a, b, freq, perf_str))

    rows.sort(key=lambda x: (-x[2], x[0], x[1]))  # frequency desc

    with open(out_path, "w", encoding="utf-8") as f:
        for a, b, freq, perf_str in rows:
            f.write(f"{a} -> {b} (frequency = {freq}, performance = {perf_str})\n")

# ---- Process all CSV files ----
pattern = os.path.join(DATA_DIR, "**/**.csv")
files = glob.glob(pattern, recursive=True)

print(f"Found files: {len(files)}")
for path in files:
    try:
        compute_dfg_and_write(path, OUTPUT_DIR)
    except Exception as e:
        print(f"skip {path}: {e}")

In [ ]:
# Variant Abstraction

TOP_K = 50  # Example number of top variants to save; update this value as needed
SEP = " -> "

def pick(colnames, *aliases):
    low = {c.lower(): c for c in colnames}
    for a in aliases:
        if a.lower() in low:
            return low[a.lower()]
    return None

def fmt_variant(v_tuple):
    return SEP.join(map(str, v_tuple))

def compute_variant_and_write(csv_path, out_dir, top_k=TOP_K):
    base = os.path.basename(csv_path)

    # extract metadata from filename
    m = re.match(r"^([A-Za-z]+)-([A-Za-z]+)-([0-9.]+)-\d+\.csv$", base)
    if m:
        dat, typ, ratio = m.group(1), m.group(2), m.group(3)
        out_name = f"Variant_{dat}_{typ}_{ratio}.txt"
    else:
        out_name = f"variant_{os.path.splitext(base)[0]}.txt"
    out_path = os.path.join(out_dir, out_name)

    # === 1) load & map columns ===
    df = pd.read_csv(csv_path)

    case_col      = pick(df.columns, "case id", "case", "caseid", "case:concept:name", "case")
    activity_col  = pick(df.columns, "activity", "concept:name", "task", "lifecycle:transition")
    timestamp_col = pick(df.columns, "timestamp", "time:timestamp", "eventtime", "time")

    assert case_col and activity_col and timestamp_col, f"[{base}] Missing required columns (case/activity/timestamp)"

    # type/ordering cleanup
    df = df[[case_col, activity_col, timestamp_col]].dropna()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    df = df.dropna(subset=[timestamp_col]).sort_values([case_col, timestamp_col], kind="mergesort")

    # === 2) variants & throughput per case ===
    case_to_seq = df.groupby(case_col)[activity_col].apply(list)
    case_times = (
        df.groupby(case_col)[timestamp_col]
          .agg(lambda s: (s.max() - s.min()).total_seconds())
          .rename("throughput_sec")
    )

    # === 3) aggregation ===
    variant_counter = Counter()
    variant_durations = defaultdict(list)
    for cid, seq in case_to_seq.items():
        var = tuple(seq)
        variant_counter[var] += 1
        variant_durations[var].append(case_times.loc[cid])

    records = []
    for v, freq in variant_counter.items():
        durs = variant_durations[v]
        avg_perf = (sum(durs) / len(durs)) if durs else None
        records.append({"variant": v, "frequency": freq, "performance_sec": avg_perf})

    res = pd.DataFrame(records).sort_values(["frequency","performance_sec"], ascending=[False, True])

    # === 4) save ===
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in res.head(top_k).iterrows():
            v_str = fmt_variant(row["variant"])
            freq  = int(row["frequency"])
            perf  = row["performance_sec"]
            perf_str = f"{perf:.2f}" if perf is not None else "NA"
            f.write(f"{v_str} (frequency = {freq} performance = {perf_str})\n")

    print(f"saved: {out_path} (variants={min(len(res), top_k)}/{len(res)})")


# ---- Process all CSV files ----
pattern = os.path.join(DATA_DIR, "**/**.csv")
files = glob.glob(pattern, recursive=True)

print(f"Found files: {len(files)}")
for path in files:
    try:
        compute_variant_and_write(path, OUTPUT_DIR, top_k=TOP_K)
    except Exception as e:
        print(f"skip {path}: {e}")


In [ ]:
# Petri Net Abstraction


def pick(cols,*alts):
    low = {c.lower(): c for c in cols}
    for a in alts:
        if a.lower() in low:
            return low[a.lower()]
    return None

def ensure_names(net):
    # place/transition nanme handling
    for i, p in enumerate(net.places):
        if not getattr(p, "name", None):
            p.name = f"p{i+1}"
    for j, t in enumerate(net.transitions):
        if not getattr(t, "name", None):
            lab = t.label if t.label else "tau"
            t.name = f"{lab}_{j+1}"

def petri_textual_abstraction(csv_path, out_dir):
    base = os.path.basename(csv_path)

    # extract metadata from filename
    m = re.match(r"^([A-Za-z]+)-([A-Za-z]+)-([0-9.]+)-\d+\.csv$", base)
    if m:
        dat, typ, ratio = m.group(1), m.group(2), m.group(3)
        out_name = f"PetriNet_{dat}_{typ}_{ratio}.txt"
    else:
        out_name = f"PetriNet_{os.path.splitext(base)[0]}.txt"
    out_path = os.path.join(out_dir, out_name)

    # --- 1) load & map standard columns ---
    df = pd.read_csv(csv_path)

    case_col      = pick(df.columns,"case id","case","caseid","case:concept:name")
    activity_col  = pick(df.columns,"activity","concept:name","task","activity name")
    timestamp_col = pick(df.columns,"timestamp","time:timestamp","eventtime","time")
    assert case_col and activity_col and timestamp_col, f"[{base}] Missing required columns (case/activity/timestamp)"

    df = df[[case_col, activity_col, timestamp_col]].dropna()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    df = df.dropna(subset=[timestamp_col]).sort_values([case_col, timestamp_col], kind="mergesort")

    counts = df[case_col].value_counts()
    df = df[df[case_col].isin(counts[counts >= 5].index)]

    # --- 2) pm4py event log conversion ---
    df_pm = df.rename(columns={
        case_col: "case:concept:name",
        activity_col: "concept:name",
        timestamp_col: "time:timestamp"
    })
    event_log = log_converter.apply(df_pm)

    # --- 3) Heuristics Miner → Petri net ---
    parameters = {
        heuristics_miner.Variants.CLASSIC.value.Parameters.DEPENDENCY_THRESH: 0.5,
    }

    res = heuristics_miner.apply(event_log, parameters=parameters)
    if isinstance(res, tuple):
        net, im, fm = res
    else:
        tree = res
        net, im, fm = pt_converter.apply(tree)

    ensure_names(net)

    # --- 4) extract structure ---
    places = [p.name for p in net.places]
    transitions = [(t.label if t.label else "tau", t.name) for t in net.transitions]
    arcs = [(getattr(a.source, "name", str(a.source)),
             getattr(a.target, "name", str(a.target))) for a in net.arcs]
    initial_marking = [f"{p.name}:{im[p]}" for p in im]
    final_marking   = [f"{p.name}:{fm[p]}" for p in fm]

    # --- 5) save textual representation ---
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("places:\n")
        f.write(", ".join(places) + "\n\n")

        f.write("transitions:\n")
        for lab, name in transitions:
            f.write(f"({lab}, {name})\n")
        f.write("\n")

        f.write("arcs:\n")
        for src, tgt in arcs:
            f.write(f"({src} -> {tgt})\n")
        f.write("\n")

        f.write("initial marking:\n")
        f.write(", ".join(initial_marking) + "\n\n")

        f.write("final marking:\n")
        f.write(", ".join(final_marking) + "\n")

    print(f"saved: {out_path} (|P|={len(places)}, |T|={len(transitions)}, |F|={len(arcs)})")

# ---- Process all CSV files ----
pattern = os.path.join(DATA_DIR, "**/**.csv")
files = glob.glob(pattern, recursive=True)

print(f"Found files: {len(files)}")
for path in files:
    try:
        petri_textual_abstraction(path, OUTPUT_DIR)
    except Exception as e:
        print(f"skip {path}: {e}")
